# 四套零售数据的五种树模型比较

按 Decision Tree → Random Forest → LightGBM → XGBoost → CatBoost 汇总训练回看、训练期滚动验证和最终 28 天测试。该 Notebook 只读取输入哈希完全一致的最新模型运行；请先依次执行前五个 Notebook。

比较反映当前 CSV、共同特征、候选参数范围和随机种子，不代表任何模型在其他数据上的普遍优劣。

## 1. 载入并验证五组实验

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
import os, sys, inspect


def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else [Path.cwd(), *Path.cwd().parents]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "notebooks"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT"
    )


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
import pandas as pd
from IPython.display import display, Code
from retail_forecast.common import *
from retail_forecast.trees import *

display(
    pd.DataFrame(
        [
            {
                "scenario": name,
                "days": len(frame),
                "start": frame.ds.min(),
                "end": frame.ds.max(),
                "open_zero_rate": float(frame.loc[frame.is_open, "y"].eq(0).mean()),
                "stockout_days": int(frame.stockout.sum()),
            }
            for name, frame in datasets.items()
        ]
    )
)

In [ ]:
tables, sources = load_comparison_results()
display(
    pd.DataFrame(
        [{"model": model, "source": str(path)} for model, path in sources.items()]
    )
)
assert list(sources) == MODEL_ORDER

## 2. 训练、验证与测试指标

In [ ]:
comparison, out = export_comparison(tables)
display(comparison.sort_values(["scenario", "split", "model"]).round(4))
display(
    comparison.loc[comparison.split.eq("test")]
    .pivot(index="scenario", columns="model", values=["MAE", "RMSE", "WAPE", "Bias"])
    .round(4)
)

## 3. 五模型性能与最终测试预测图

In [ ]:
plot_comparison(tables, comparison, out)

## 4. 可比较的特征重要性

不同库的原生 gain、impurity 和 feature importance 定义不同，不能直接比较。这里统一查看滚动验证数据被打乱后引起的 MAE 增量；负值表示该特征在有限验证样本上可能增加噪声。

In [ ]:
comparable = tables["importance"].loc[
    tables["importance"].importance_type.eq("permutation_MAE_increase")
]
display(
    comparable.sort_values(
        ["scenario", "model", "importance"], ascending=[True, True, False]
    )
    .groupby(["scenario", "model"])
    .head(10)
    .round(4)
)

## 5. 选定参数与交付边界

In [ ]:
display(tables["parameters"])
assert set(comparison.model) == set(MODEL_ORDER)
assert set(comparison.scenario) == set(SCENARIOS)
assert set(comparison.split) == {"train_hindcast", "validation", "test"}
print("Exports:", out.resolve())